In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install dagshub

In [ ]:
import dagshub

In [ ]:
!pip install mlflow

# 1. Data Loading and Initial Setup


In [ ]:
import mlflow
import dagshub
dagshub.init(repo_owner='vasiliDandurishvili', repo_name='house_prices', mlflow=True)

In [ ]:
import numpy as np
import pandas as pd
train = pd.read_csv('/kaggle/input/datasets/vasilidandurishvili/all-data/train.csv')
X_test = pd.read_csv('/kaggle/input/datasets/vasilidandurishvili/all-data/test.csv')
test_ids = X_test["Id"]

print(train.shape)
train.head()

In [ ]:
os.listdir('/kaggle/input')

In [ ]:
train.head()
train.shape
train.info()
train.describe()

In [ ]:
mlflow.log_param("model_type", "LinearRegression")

In [ ]:
mlflow.log_param("new_param", 1)

In [ ]:
mlflow.end_run()

In [ ]:
mlflow.end_run()

In [ ]:
mlflow.log_param("new_param", 1)

In [ ]:
train.head()

In [ ]:
train.head()

# 2. Train/Validation Split

In [ ]:
X = train.drop(columns = ["SalePrice"])
y = train["SalePrice"]

y = np.log1p(y) 

# 3. Feature Engineering - String Columns Encoding

In [ ]:


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

mlflow.end_run()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

string_cols = X_train.select_dtypes(include=['object']).columns.tolist()

remaining_string_cols = [col for col in string_cols]
print(f"\nRemaining string columns to encode with numbers: {len(remaining_string_cols)}")

for col in remaining_string_cols:
    # Get unique values from training set only (excluding NA)
    unique_values = X_train[col].dropna().unique()
    
    # Create mapping: string -> number (0,1,2,...)
    mapping = {value: idx for idx, value in enumerate(unique_values)}
    
    # Apply mapping to train, validation, and test
    X_train[col] = X_train[col].map(mapping)
    X_val[col] = X_val[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)

print(f"\nExample of encoded values:")
for col in string_cols[:3]:
    print(f"{col}: {X_train[col].dropna().unique()[:5]}")

In [ ]:
X_train.head()

# 4. Data Cleaning - Dropping High N/A Columns

In [ ]:
import pandas as pd
import numpy as np

print("="*60)
print("DROPPING COLUMNS WITH > 40% N/A VALUES")
print("="*60)


na_percentage = (X_train.isnull().sum() / len(X_train)) * 100


cols_to_drop = na_percentage[na_percentage > 40].index.tolist()

print(f"\nColumns to drop: {len(cols_to_drop)}")
print(f"\nColumns with > 40% N/A values:")
print("-"*50)

for col in cols_to_drop:
    print(f"  {col:<25} {na_percentage[col]:>6.1f}%")


X_train.drop(columns=cols_to_drop, inplace=True, errors='ignore')
X_val.drop(columns=cols_to_drop, inplace=True, errors='ignore')
X_test.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f"\nNew shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")
print("\nComplete!")

In [ ]:
X_train.info()

# 5. Feature Selection - Correlation Analysis

In [ ]:

correlations = X_train.corrwith(y_train).sort_values(ascending=False)

print("="*60)
print("CORRELATION WITH TARGET (SalePrice)")
print("="*60)
print(correlations.head(20))

# 5.1 Correlation Threshold Testing (0.7, 0.75, 0.8, 0.85, 0.9, 0.95)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

print("="*70)
print("CORRELATION THRESHOLD ANALYSIS")
print("="*70)


thresholds = [0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
results = []


X_train_orig = X_train.copy()
X_val_orig = X_val.copy()

for thresh in thresholds:
    print(f"\n{'='*60}")
    print(f"ANALYZING THRESHOLD: Correlation > {thresh}")
    print(f"{'='*60}")
    

    corr_matrix = X_train_orig.corr().abs()
    
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    
    cols_to_drop = set()
    
    for column in upper.columns:
        high_corr = upper[column][upper[column] > thresh].index.tolist()
        
        if high_corr:
            corr_with_target = X_train_orig.corrwith(y_train).abs()
            
            for corr_col in high_corr:
                if corr_with_target[column] < corr_with_target[corr_col]:
                    cols_to_drop.add(column)
                else:
                    cols_to_drop.add(corr_col)
    
    cols_to_drop = list(cols_to_drop)
    
    print(f"Columns that WOULD be dropped: {len(cols_to_drop)}")
    if len(cols_to_drop) > 0 and len(cols_to_drop) <= 15:
        for col in cols_to_drop:
            max_corr = 0
            for other in upper.columns:
                if other != col and abs(corr_matrix.loc[col, other]) > max_corr:
                    max_corr = abs(corr_matrix.loc[col, other])
            print(f"  - {col} (max correlation: {max_corr:.3f})")
    elif len(cols_to_drop) > 15:
        print(f"  (showing first 15)")
        for col in cols_to_drop[:15]:
            print(f"  - {col}")
        print(f"  ... and {len(cols_to_drop)-15} more")

    
    X_train_test = X_train_orig.drop(columns=cols_to_drop, errors='ignore')
    X_val_test = X_val_orig.drop(columns=cols_to_drop, errors='ignore')

    
    X_train_test = X_train_test.fillna(0)
    X_val_test = X_val_test.fillna(0)
    
    
    print("Training RandomForestRegressor...")
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    model.fit(X_train_test, y_train)
    
    
    y_pred = model.predict(X_val_test)
    
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    
    print(f"\nSCORES (after dropping correlated columns):")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²: {r2:.4f}")
    
    
    X_train_orig_filled = X_train_orig.fillna(0)
    X_val_orig_filled = X_val_orig.fillna(0)
    
    model_original = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    model_original.fit(X_train_orig_filled, y_train)
    y_pred_original = model_original.predict(X_val_orig_filled)
    
    rmse_original = np.sqrt(mean_squared_error(y_val, y_pred_original))
    r2_original = r2_score(y_val, y_pred_original)
    
    print(f"\nSCORES (original data - no drops):")
    print(f"  RMSE: {rmse_original:.4f}")
    print(f"  R²: {r2_original:.4f}")
    
    print(f"\nIMPROVEMENT:")
    print(f"  RMSE change: {rmse_original - rmse:.4f}")
    print(f"  R² change: {r2 - r2_original:.4f}")
    
    
    results.append({
        'threshold': thresh,
        'cols_would_drop': len(cols_to_drop),
        'rmse_after_drop': rmse,
        'r2_after_drop': r2,
        'rmse_original': rmse_original,
        'r2_original': r2_original,
        'improvement': rmse_original - rmse
    })


results_df = pd.DataFrame(results)


best_improvement = results_df.loc[results_df['improvement'].idxmax()]

print(f"\n{'='*70}")
print("SUMMARY RESULTS")
print(f"{'='*70}")
print(results_df.to_string(index=False))

print(f"\n{'='*70}")
print(f"BEST THRESHOLD: {best_improvement['threshold']}")
print(f"  Improvement: {best_improvement['improvement']:.4f}")
print(f"  Columns to drop: {best_improvement['cols_would_drop']}")
print(f"{'='*70}")


results_df.to_csv("correlation_analysis_results.csv", index=False)
print("\nSaved results to 'correlation_analysis_results.csv'")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(results_df['threshold'], results_df['rmse_original'], 'bo-', label='Original (no drop)', linewidth=2)
axes[0].plot(results_df['threshold'], results_df['rmse_after_drop'], 'ro-', label='After dropping correlated', linewidth=2)
axes[0].set_xlabel('Correlation Threshold')
axes[0].set_ylabel('RMSE')
axes[0].set_title('RMSE Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(results_df['threshold'], results_df['improvement'], 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Correlation Threshold')
axes[1].set_ylabel('RMSE Improvement')
axes[1].set_title('Improvement by Dropping Correlated Features')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='red', linestyle='--')

plt.tight_layout()
plt.savefig('correlation_analysis.png')
plt.show()

print("\nAnalysis complete!")


# 5.2 Applying Best Threshold (0.85) to Drop Correlated Features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

print("="*70)
print("DATA PREPROCESSING PIPELINE")
print("="*70)


print("\n2. Removing highly correlated features (>=0.85)...")

corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_with_target = X_train.corrwith(y_train).abs()

cols_to_drop_corr = set()

for column in upper.columns:
    high_corr = upper[column][upper[column] >= 0.85].index.tolist()
    
    if high_corr:
        for corr_col in high_corr:
            if corr_with_target[column] < corr_with_target[corr_col]:
                cols_to_drop_corr.add(column)
            else:
                cols_to_drop_corr.add(corr_col)

cols_to_drop_corr = list(cols_to_drop_corr)
print(f"   Dropped {len(cols_to_drop_corr)} correlated columns")

X_train.drop(columns=cols_to_drop_corr, inplace=True, errors='ignore')
X_val.drop(columns=cols_to_drop_corr, inplace=True, errors='ignore')
X_test.drop(columns=cols_to_drop_corr, inplace=True, errors='ignore')


print("\n3. Filling NA values...")

# Columns that were originally strings (now encoded as numbers) -> fill with MODE
for col in string_cols:  # string_cols was defined in your encoding cell
    if col in X_train.columns and X_train[col].isnull().any():
        mode_val = X_train[col].mode()
        mode_val = mode_val[0] if len(mode_val) > 0 else 0
        X_train[col] = X_train[col].fillna(mode_val)
        X_val[col]   = X_val[col].fillna(mode_val)
        X_test[col]  = X_test[col].fillna(mode_val)

# Columns that were originally integers -> fill with MEDIAN
int_cols = X_train.select_dtypes(include=['int64', 'int32']).columns.tolist()
for col in int_cols:
    if X_train[col].isnull().any():
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col]   = X_val[col].fillna(median_val)
        X_test[col]  = X_test[col].fillna(median_val)

# Columns that were originally floats -> fill with MEDIAN
float_cols = X_train.select_dtypes(include=['float64', 'float32']).columns.tolist()
float_cols = [c for c in float_cols if c not in string_cols]
for col in float_cols:
    if X_train[col].isnull().any():
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col]   = X_val[col].fillna(median_val)
        X_test[col]  = X_test[col].fillna(median_val)

# Check if any NA remain
remaining_na = X_train.isnull().sum().sum()
print(f"\n   Remaining NA values: {remaining_na}")

if remaining_na == 0:
    print("   SUCCESS: All NA values filled!")
else:
    print("   WARNING: Some NA values still remain")


print("\n4. Final data shapes:")
print(f"   X_train: {X_train.shape}")
print(f"   X_val: {X_val.shape}")
print(f"   X_test: {X_test.shape}")

# Quick test with Random Forest
print("\n5. Quick model test...")
model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

if y_train.mean() < 100:
    y_pred_orig = np.expm1(y_pred)
    y_val_orig = np.expm1(y_val)
    rmse_orig = np.sqrt(mean_squared_error(y_val_orig, y_pred_orig))
    print(f"   RMSE (log scale): {rmse:.4f}")
    print(f"   RMSE (original price): ${rmse_orig:.2f}")
else:
    print(f"   RMSE: ${rmse:.2f}")

print("\n" + "="*70)
print("PREPROCESSING COMPLETE!")
print("="*70)

# 6. Feature Selection - RFE (Recursive Feature Elimination)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


ENABLE_MLFLOW = False  

if ENABLE_MLFLOW:
    import mlflow
    mlflow.end_run()
    mlflow.start_run(run_name="rfe_select_40_features")

print("="*70)
print("="*70)


features_before = X_train.shape[1]
print(f"\nFeatures before RFE: {features_before}")


print("\nRunning RFE to select 40 features...")
selector = RFE(
    RandomForestRegressor(n_estimators=100, random_state=42), 
    n_features_to_select=40,
    step=5
)


selector.fit(X_train, y_train)


selected_features = X_train.columns[selector.support_].tolist()
print(f"\nSelected {len(selected_features)} features")


X_train = X_train[selected_features]
X_val = X_val[selected_features]
X_test = X_test[selected_features]

print(f"\nNew shapes after RFE:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")

print("\n" + "="*70)
print("RFE COMPLETE!")
print("="*70)

In [ ]:
!pip install xgboost

# 7. Model Training - Random Forest Baseline

In [ ]:
# mlflow.end_run()

# with mlflow.start_run(run_name="random forest: remove corr >= 0.85 and fill categorial N/A with mode and numrical with median, n_estimator: 15"):
#     model = RandomForestRegressor(n_estimators=15, random_state=42)  
#     model.fit(X_train, y_train)
    
#     preds = model.predict(X_val)
#     rmse = np.sqrt(mean_squared_error(y_val, preds))
    
    
#     mlflow.log_metric("rmse", rmse)
#     mlflow.sklearn.log_model(model, "model")

model = RandomForestRegressor(n_estimators=15, random_state=42)  
model.fit(X_train, y_train)

preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
    
print("Test RMSE:", rmse)

# 8. Model Training - XGBoost (Tuned)

In [ ]:
import xgboost as xgb
with mlflow.start_run(run_name="FINAL!!! best xgb_experiment n_est=1000, max_depth = 5, learning_rate = 0.04, with "):
     model = xgb.XGBRegressor(
         n_estimators=1000,
         max_depth=5,
         learning_rate=0.04, # 0.05
         random_state=42,
         n_jobs=-1
     )
     model.fit(X_train, y_train)
     preds = model.predict(X_val)
     rmse = np.sqrt(mean_squared_error(y_val, preds))
     print("Test RMSE:", rmse)
     
     mlflow.log_metric("rmse", rmse)
     mlflow.log_param("n_estimators", 1000)
     mlflow.log_param("model_type", "XGBoost")

     mlflow.xgboost.log_model(model, "model")



test_preds = model.predict(X_test)

test_preds = np.expm1(test_preds)  # ADD THIS

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_preds
})

submission.to_csv("submission.csv", index = False)

